In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from wordcloud import WordCloud
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

print(train.head())

In [ ]:
print(train.shape)
print(test.shape)


In [ ]:
print(train.head())
print(train.info())
print(train['target'].value_counts())

In [ ]:

train['keyword'].fillna('none', inplace=True)
train['location'].fillna('unknown', inplace=True)
test['keyword'].fillna('none', inplace=True)
test['location'].fillna('unknown', inplace=True)

train['text'] = train['text'].str.lower()
test['text'] = test['text'].str.lower()

In [ ]:

# Identify tweets with the same text but different labels
dupe_conflict = train.groupby('text')['target'].nunique()
conflicting_texts = dupe_conflict[dupe_conflict > 1].index

print(f"Number of tweets with conflicting labels: {len(conflicting_texts)}")

In [ ]:
# clean_train = train[~train['text'].isin(conflicting_texts)].reset_index(drop=True)
# print(f"Train set size after removing ambiguous rows: {clean_train.shape}")

# For tweets with conflicting classes, keep the target with the most votes
mode_target = train.groupby('text')['target'].agg(lambda x: x.mode().iloc[0])
deduped_train = train.drop_duplicates(subset=['text'])
deduped_train['target'] = deduped_train['text'].map(mode_target)
print(f"Train set size after deduplication: {deduped_train.shape}")

# train = clean_train
train = deduped_train

In [ ]:
print("\nExample disaster tweets:\n")
print(train[train['target']==1][['text', 'keyword']].sample(5, random_state=42))

In [ ]:
print("\nExample non-disaster tweets:\n")
print(train[train['target']==0][['text', 'keyword']].sample(5, random_state=42))

In [ ]:
print("\nCommon keywords for disaster tweets:")
print(train[train['target']==1]['keyword'].value_counts().head(10))

print("\nCommon keywords for non-disaster tweets:")
print(train[train['target']==0]['keyword'].value_counts().head(10))

In [ ]:
print(train.isna().sum())


In [ ]:


disaster_text = " ".join(train[train['target']==1]['text'])
not_disaster_text = " ".join(train[train['target']==0]['text'])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.title("Disaster Tweets")
plt.imshow(WordCloud(width=800, height=800).generate(disaster_text))
plt.axis('off')

plt.subplot(1,2,2)
plt.title("Not Disaster Tweets")
plt.imshow(WordCloud(width=800, height=800).generate(not_disaster_text))
plt.axis('off')
plt.show()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text) #remove urls
    text = re.sub(r"[^a-zA-Z\s]", "", text) #remove special chars
    return text

train['text'] = train['text'].apply(clean_text)
test['text'] = test['text'].apply(clean_text)

In [ ]:

max_words = 10000
max_len = 40

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train['text'])

X_train = tokenizer.texts_to_sequences(train['text'])
X_train = pad_sequences(X_train, maxlen=max_len, padding='post')

X_test = tokenizer.texts_to_sequences(test['text'])
X_test = pad_sequences(X_test, maxlen=max_len, padding='post')


In [ ]:
y_train = train['target']


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

model = Sequential([
    Embedding(max_words, 64, ),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid'),
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',    
    patience=3,            
    restore_best_weights=True 
)

history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=2,
    callbacks=[early_stop]
)


In [ ]:
# predictions = model.predict(X_test)
# pred_labels = (predictions > 0.5).astype(int).flatten()

# submission = pd.DataFrame({'id': test['id'], 'target': pred_labels})
# submission.to_csv('submission.csv', index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Fit TF-IDF on both train and test sets for vocab consistency
all_text = pd.concat([train['text'], test['text']])
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
tfidf.fit(all_text)

X_train_tfidf = tfidf.transform(train['text'])
X_test_tfidf = tfidf.transform(test['text'])

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
lr_preds_prob = lr.predict_proba(X_test_tfidf)[:,1]


In [ ]:
nn_preds_prob = model.predict(X_test).flatten()


In [ ]:
ensemble_prob = (nn_preds_prob + lr_preds_prob) / 2
ensemble_preds = (ensemble_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'target': ensemble_preds
})
submission.to_csv('submission_ensemble.csv', index=False)
